<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_06-4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 지능시스템 실습 4: Retrieval 평가 지표

이번 실습에서는 RAG의 **검색기(Retriever)** 가 얼마나 잘 동작하는지 평가합니다.

### 학습 목표
- Hit Rate@k를 계산할 수 있다.
- Recall@k를 계산할 수 있다.
- MRR(Mean Reciprocal Rank)을 계산할 수 있다.
- nDCG@k를 계산할 수 있다.
- 각 지표가 무엇을 의미하는지 비교할 수 있다.

> 핵심 질문: **정답 문서가 검색 결과 안에 잘 들어왔는가?**

## 1. 실습용 검색 결과 만들기

각 질의마다 검색기가 문서를 높은 순서대로 반환했다고 가정합니다.

예를 들어:

- `q1`의 정답 문서: `A`, `B`
- 검색 결과: `A`, `D`, `C`, `B`, `E`

이 경우 정답 문서 `A`는 1위, `B`는 4위에 있습니다.

In [1]:
# 각 질의의 정답 관련 문서
ground_truth = {
    "q1": {"A", "B"},
    "q2": {"C"},
    "q3": {"D", "E"},
    "q4": {"F"}
}

# 검색기가 반환한 문서 순위
ranked_results = {
    "q1": ["A", "D", "C", "B", "E"],
    "q2": ["A", "B", "C", "D", "E"],
    "q3": ["A", "D", "B", "C", "E"],
    "q4": ["A", "B", "C", "D", "E"]
}

for qid in ground_truth:
    print(qid)
    print("정답 문서:", ground_truth[qid])
    print("검색 결과:", ranked_results[qid])
    print()

q1
정답 문서: {'B', 'A'}
검색 결과: ['A', 'D', 'C', 'B', 'E']

q2
정답 문서: {'C'}
검색 결과: ['A', 'B', 'C', 'D', 'E']

q3
정답 문서: {'D', 'E'}
검색 결과: ['A', 'D', 'B', 'C', 'E']

q4
정답 문서: {'F'}
검색 결과: ['A', 'B', 'C', 'D', 'E']



# 2. Hit Rate@k

Hit Rate@k는 **상위 k개 검색 결과 안에 정답 문서가 하나라도 있는가?** 를 봅니다.

- 있으면 `1`
- 없으면 `0`

### 예
정답 문서가 `{A, B}`이고 Top-3가 `[A, D, C]`라면  
정답 `A`가 포함되어 있으므로:

**Hit Rate@3 = 1**

In [2]:
def hit_rate_at_k(relevant_docs, retrieved_docs, k):
    """
    상위 k개 안에 정답 문서가 하나라도 있으면 1,
    하나도 없으면 0을 반환합니다.
    """

    top_k = retrieved_docs[:k]

    # 관련 문서와 Top-k 결과의 교집합
    hits = set(top_k) & set(relevant_docs)

    return 1 if len(hits) > 0 else 0

In [3]:
# q1에 대해 직접 확인
qid = "q1"
k = 3

score = hit_rate_at_k(
    ground_truth[qid],
    ranked_results[qid],
    k
)

print("질의:", qid)
print("정답:", ground_truth[qid])
print("Top-3:", ranked_results[qid][:3])
print("Hit Rate@3 =", score)

질의: q1
정답: {'B', 'A'}
Top-3: ['A', 'D', 'C']
Hit Rate@3 = 1


## 여러 질의의 평균 Hit Rate@k 계산

In [4]:
def mean_hit_rate_at_k(ground_truth, ranked_results, k):
    """
    여러 질의의 Hit Rate@k 평균을 계산합니다.
    """

    scores = []

    for qid in ground_truth:
        score = hit_rate_at_k(
            ground_truth[qid],
            ranked_results[qid],
            k
        )
        scores.append(score)

    return sum(scores) / len(scores)


for k in [1, 3, 5]:
    print(
        f"Mean Hit Rate@{k} =",
        round(mean_hit_rate_at_k(ground_truth, ranked_results, k), 4)
    )

Mean Hit Rate@1 = 0.25
Mean Hit Rate@3 = 0.75
Mean Hit Rate@5 = 0.75


# 3. Recall@k

Recall@k는 **전체 정답 문서 중에서 Top-k 안에 몇 개를 찾았는가?** 를 봅니다.

### 식

\[
Recall@k =
rac{	ext{Top-k 안에서 찾은 정답 문서 수}}
{	ext{전체 정답 문서 수}}
\]

### 예
정답 문서가 `{A, B, C}`이고 Top-3가 `[A, D, E]`라면:

- 찾은 정답: `A` 1개
- 전체 정답: 3개

따라서:

**Recall@3 = 1/3**

In [5]:
def recall_at_k(relevant_docs, retrieved_docs, k):
    """
    전체 정답 문서 중 Top-k 안에서 찾은 비율을 계산합니다.
    """

    top_k = retrieved_docs[:k]

    # Top-k 안에서 찾은 정답 문서
    hits = set(top_k) & set(relevant_docs)

    # 정답 문서가 없는 경우를 방지
    if len(relevant_docs) == 0:
        return 0.0

    return len(hits) / len(relevant_docs)

In [6]:
# q1에 대해 Recall@3 계산
qid = "q1"
k = 3

score = recall_at_k(
    ground_truth[qid],
    ranked_results[qid],
    k
)

print("질의:", qid)
print("정답:", ground_truth[qid])
print("Top-3:", ranked_results[qid][:3])
print("Recall@3 =", round(score, 4))

질의: q1
정답: {'B', 'A'}
Top-3: ['A', 'D', 'C']
Recall@3 = 0.5


## 여러 질의의 평균 Recall@k 계산

In [7]:
def mean_recall_at_k(ground_truth, ranked_results, k):
    """
    여러 질의의 Recall@k 평균을 계산합니다.
    """

    scores = []

    for qid in ground_truth:
        score = recall_at_k(
            ground_truth[qid],
            ranked_results[qid],
            k
        )
        scores.append(score)

    return sum(scores) / len(scores)


for k in [1, 3, 5]:
    print(
        f"Mean Recall@{k} =",
        round(mean_recall_at_k(ground_truth, ranked_results, k), 4)
    )

Mean Recall@1 = 0.125
Mean Recall@3 = 0.5
Mean Recall@5 = 0.75


# 4. MRR (Mean Reciprocal Rank)

MRR은 **첫 번째 정답 문서가 몇 위에 나왔는가?** 를 평가합니다.

질의 하나의 Reciprocal Rank는:

\[
RR = rac{1}{r}
\]

여기서 `r`은 첫 번째 관련 문서의 순위입니다.

예:

- 정답이 1위 → `1/1 = 1.0`
- 정답이 2위 → `1/2 = 0.5`
- 정답이 5위 → `1/5 = 0.2`

MRR은 여러 질의의 RR 평균입니다.

In [8]:
def reciprocal_rank(relevant_docs, retrieved_docs):
    """
    첫 번째 정답 문서의 순위를 찾아 1/rank를 반환합니다.
    정답 문서가 검색되지 않으면 0을 반환합니다.
    """

    # rank는 1부터 시작
    for rank, doc_id in enumerate(retrieved_docs, start=1):

        if doc_id in relevant_docs:
            return 1 / rank

    return 0.0

In [9]:
# 각 질의의 Reciprocal Rank 확인
for qid in ground_truth:
    rr = reciprocal_rank(
        ground_truth[qid],
        ranked_results[qid]
    )

    print(
        qid,
        "정답 =", ground_truth[qid],
        "검색 =", ranked_results[qid],
        "RR =", round(rr, 4)
    )

q1 정답 = {'B', 'A'} 검색 = ['A', 'D', 'C', 'B', 'E'] RR = 1.0
q2 정답 = {'C'} 검색 = ['A', 'B', 'C', 'D', 'E'] RR = 0.3333
q3 정답 = {'D', 'E'} 검색 = ['A', 'D', 'B', 'C', 'E'] RR = 0.5
q4 정답 = {'F'} 검색 = ['A', 'B', 'C', 'D', 'E'] RR = 0.0


In [10]:
def mean_reciprocal_rank(ground_truth, ranked_results):
    """
    여러 질의의 Reciprocal Rank 평균 = MRR
    """

    scores = []

    for qid in ground_truth:
        rr = reciprocal_rank(
            ground_truth[qid],
            ranked_results[qid]
        )
        scores.append(rr)

    return sum(scores) / len(scores)


mrr = mean_reciprocal_rank(
    ground_truth,
    ranked_results
)

print("MRR =", round(mrr, 4))

MRR = 0.4583


# 5. nDCG@k

nDCG는 문서마다 관련성이 다를 때 유용합니다.

예를 들어 관련성을 다음처럼 줄 수 있습니다.

- `0`: 관련 없음
- `1`: 조금 관련 있음
- `2`: 매우 관련 있음
- `3`: 가장 관련 있음

nDCG는 두 가지를 함께 봅니다.

1. 관련성이 높은 문서가 위에 있는가?
2. 높은 순위에 있을수록 더 큰 점수를 주는가?

즉, **전체 순위 품질**을 평가합니다.

## 5-1. graded relevance 데이터 만들기

In [11]:
# 문서별 관련성 점수
# 숫자가 클수록 더 관련 있는 문서
graded_relevance = {
    "q1": {"A": 3, "B": 2, "C": 1, "D": 0, "E": 0},
    "q2": {"A": 0, "B": 1, "C": 3, "D": 0, "E": 0},
    "q3": {"A": 0, "B": 1, "C": 0, "D": 3, "E": 2},
    "q4": {"A": 0, "B": 0, "C": 0, "D": 0, "E": 0, "F": 3}
}

print("q1 관련성 점수:")
print(graded_relevance["q1"])

q1 관련성 점수:
{'A': 3, 'B': 2, 'C': 1, 'D': 0, 'E': 0}


## 5-2. DCG 계산

DCG는 순위가 내려갈수록 관련성 점수를 할인합니다.

여기서는 많이 사용하는 다음 식을 사용합니다.

\[
DCG@k =
\sum_{i=1}^{k}
rac{2^{rel_i}-1}
{\log_2(i+1)}
\]

- `rel_i`: i번째 문서의 관련성 점수
- 높은 순위일수록 할인 폭이 작습니다.

In [14]:
import math

def dcg_at_k(relevance_scores, k):
    """
    relevance_scores:
    검색 순서대로 정렬된 관련성 점수 목록

    예:
    [3, 0, 1, 2, 0]
    """

    dcg = 0.0

    for i, rel in enumerate(relevance_scores[:k], start=1):

        # 높은 순위일수록 더 큰 가중치
        gain = (2 ** rel - 1) / math.log2(i + 1)

        dcg += gain

    return dcg

In [15]:
# q1 검색 순서의 관련성 점수 만들기
qid = "q1"

relevance_in_rank = [
    graded_relevance[qid].get(doc_id, 0)
    for doc_id in ranked_results[qid]
]

print("검색 결과:", ranked_results[qid])
print("관련성 점수:", relevance_in_rank)

dcg = dcg_at_k(relevance_in_rank, k=5)

print("DCG@5 =", round(dcg, 4))

검색 결과: ['A', 'D', 'C', 'B', 'E']
관련성 점수: [3, 0, 1, 2, 0]
DCG@5 = 8.792


## 5-3. IDCG와 nDCG 계산

DCG는 질의마다 정답 개수와 관련성 점수 분포가 다르기 때문에 직접 비교하기 어렵습니다.

그래서 **가장 이상적인 순서의 DCG(IDCG)** 로 나누어 정규화합니다.

\[
nDCG@k = rac{DCG@k}{IDCG@k}
\]

일반적으로 nDCG는 0~1 사이이며, **1에 가까울수록 이상적인 순위**에 가깝습니다.

In [16]:
def ndcg_at_k(relevance_dict, retrieved_docs, k):
    """
    한 질의의 nDCG@k를 계산합니다.

    relevance_dict:
        {"A": 3, "B": 2, ...}

    retrieved_docs:
        ["A", "D", "C", ...]
    """

    # 실제 검색 순서의 관련성 점수
    actual_scores = [
        relevance_dict.get(doc_id, 0)
        for doc_id in retrieved_docs[:k]
    ]

    # 실제 DCG
    dcg = dcg_at_k(actual_scores, k)

    # 이상적인 순서:
    # 관련성 점수를 큰 값부터 정렬
    ideal_scores = sorted(
        relevance_dict.values(),
        reverse=True
    )[:k]

    # 이상적인 DCG
    idcg = dcg_at_k(ideal_scores, k)

    # 관련 문서가 하나도 없으면 0
    if idcg == 0:
        return 0.0

    return dcg / idcg

In [17]:
# 각 질의의 nDCG@5 계산
for qid in graded_relevance:
    score = ndcg_at_k(
        graded_relevance[qid],
        ranked_results[qid],
        k=5
    )

    print(
        qid,
        "nDCG@5 =",
        round(score, 4)
    )

q1 nDCG@5 = 0.936
q2 nDCG@5 = 0.5413
q3 nDCG@5 = 0.647
q4 nDCG@5 = 0.0


# 6. 모든 평가 지표를 한 번에 비교하기

In [19]:
import pandas as pd

rows = []

for qid in ground_truth:

    row = {
        "Query": qid,
        "Hit@3": hit_rate_at_k(
            ground_truth[qid],
            ranked_results[qid],
            3
        ),
        "Recall@3": recall_at_k(
            ground_truth[qid],
            ranked_results[qid],
            3
        ),
        "RR": reciprocal_rank(
            ground_truth[qid],
            ranked_results[qid]
        ),
        "nDCG@5": ndcg_at_k(
            graded_relevance[qid],
            ranked_results[qid],
            5
        )
    }

    rows.append(row)

df = pd.DataFrame(rows)

# 보기 좋게 반올림
for col in ["Recall@3", "RR", "nDCG@5"]:
    df[col] = df[col].round(4)

df

,Query,Hit@3,Recall@3,RR,nDCG@5
0,q1,1,0.5,1.0000,0.9360
1,q2,1,1.0,0.3333,0.5413
2,q3,1,0.5,0.5000,0.6470
3,q4,0,0.0,0.0000,0.0000


# 7. 지표 해석하기

각 지표는 보는 관점이 다릅니다.

| 지표 | 핵심 질문 |
|---|---|
| Hit Rate@k | Top-k 안에 정답이 하나라도 있는가? |
| Recall@k | 전체 정답 중 몇 개를 찾았는가? |
| MRR | 첫 번째 정답이 얼마나 위에 있는가? |
| nDCG@k | 관련성이 높은 문서가 위에 잘 배치되었는가? |

### 실무적인 해석 예

- Recall@k가 낮다  
  → 검색기가 정답 문서를 충분히 찾지 못함

- Recall@10은 높은데 MRR이 낮다  
  → 정답은 찾았지만 순위가 너무 아래에 있음  
  → **Reranker** 개선을 고려할 수 있음

- nDCG가 낮다  
  → 전체 순위 품질이 좋지 않거나, 강하게 관련된 문서가 아래에 있을 수 있음

# 실습 정리

이번 실습의 핵심은 다음과 같습니다.

### Hit Rate@k
**정답을 하나라도 찾았는가?**

### Recall@k
**전체 정답 중 얼마나 많이 찾았는가?**

### MRR
**첫 번째 정답을 얼마나 위에 올렸는가?**

### nDCG@k
**관련성이 높은 문서를 전체적으로 얼마나 좋은 순서에 배치했는가?**

---

## RAG 디버깅 관점

LLM의 답변이 틀렸을 때 바로 생성 모델만 의심하면 안 됩니다.

먼저:

**정답 문서가 검색되었는가? → 검색 순위가 적절한가? → 필요한 Context가 LLM에 전달되었는가?**

를 단계적으로 확인해야 합니다.